# NB01 — API Latency Benchmarks

Measures **p50 / p90 / p95 / p99 latency**, min, max, and error rate for each key QCanvas API endpoint.

Also measures **cold vs. warm** response time to quantify the Redis caching benefit.

**Requires:** Backend running at `http://localhost:8000`  
**Output:** `../results/api/latency.csv`, `../results/api/cold_warm.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'scripts'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
from pathlib import Path

from api_timing import (
    get_auth_token,
    auth_headers,
    benchmark_endpoint,
    cold_vs_warm,
    get_endpoint_catalogue,
)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (12, 5)})

RESULTS_DIR = Path('../results/api')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_RUNS = 20   # number of measured calls per endpoint
WARMUP = 3    # discarded warmup calls

print('✓ Imports OK')

## 1 — Authentication

Obtain a JWT token for endpoints that require auth.

In [ ]:
token = get_auth_token()
if token:
    print(f'✓ Authenticated (token: {token[:20]}...)')
else:
    print('⚠  No token obtained — protected endpoints will run unauthenticated')
hdrs = auth_headers(token)

## 2 — Benchmark All Endpoints

In [ ]:
catalogue = get_endpoint_catalogue(token)
summaries = []

for ep in catalogue:
    print(f'Benchmarking: {ep["label"]} ... ', end='')
    result = benchmark_endpoint(
        ep['method'], ep['url'],
        n=N_RUNS,
        warmup=WARMUP,
        headers=ep.get('headers', {}),
        json=ep.get('json'),
        label=ep['label'],
    )
    summaries.append(result)
    print(f"p50={result['p50_ms']:.1f}ms  p95={result['p95_ms']:.1f}ms  err={result['error_rate']:.1%}")

df = pd.DataFrame([{k: v for k, v in s.items() if k != 'raw_latencies'} for s in summaries])
df.to_csv(RESULTS_DIR / 'latency.csv', index=False)
print('\n✓ Saved latency.csv')
df

## 3 — Latency Percentile Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

short_labels = [ep['label'].replace('POST ', '').replace('GET ', '') for ep in catalogue]

# p50 vs p95 grouped bar
x = np.arange(len(df))
w = 0.35
axes[0].bar(x - w/2, df['p50_ms'], w, label='p50', color='steelblue', alpha=0.85)
axes[0].bar(x + w/2, df['p95_ms'], w, label='p95', color='tomato', alpha=0.85)
axes[0].axhline(2000, color='red', linestyle='--', linewidth=1, label='2s target')
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_labels, rotation=35, ha='right', fontsize=8)
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('p50 vs p95 Latency per Endpoint')
axes[0].legend()

# p99 bar
axes[1].bar(x, df['p99_ms'], color='orchid', alpha=0.85)
axes[1].axhline(2000, color='red', linestyle='--', linewidth=1, label='2s target')
axes[1].set_xticks(x)
axes[1].set_xticklabels(short_labels, rotation=35, ha='right', fontsize=8)
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('p99 Latency (Tail)')
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'latency_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved latency_bars.png')

## 4 — Latency Distribution (Box Plot)

In [ ]:
raw_data = []
for s in summaries:
    for lat in s['raw_latencies']:
        raw_data.append({'endpoint': s['endpoint'], 'latency_ms': lat})
df_raw = pd.DataFrame(raw_data)

fig, ax = plt.subplots(figsize=(14, 6))
order = df.sort_values('p95_ms', ascending=False)['endpoint'].tolist()
sns.boxplot(data=df_raw, x='endpoint', y='latency_ms', order=order, ax=ax, palette='coolwarm')
ax.axhline(2000, color='red', linestyle='--', linewidth=1.2, label='2s target')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
ax.set_title('Latency Distribution per Endpoint (Box Plot)')
ax.set_ylabel('Latency (ms)')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'latency_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved latency_boxplot.png')

## 5 — Cold vs. Warm (Redis Caching Benefit)

In [ ]:
import json

cold_warm_targets = [
    {
        'label': 'Converter (Cirq Bell)',
        'method': 'POST',
        'url': 'http://localhost:8000/api/converter/convert',
        'json': {
            'code': 'import cirq\nq0, q1 = cirq.LineQubit.range(2)\ncircuit = cirq.Circuit([cirq.H(q0), cirq.CNOT(q0, q1), cirq.measure(q0, q1, key="r")])\n',
            'framework': 'cirq',
        },
    },
    {
        'label': 'Simulator (statevector)',
        'method': 'POST',
        'url': 'http://localhost:8000/api/simulator/simulate',
        'json': {
            'qasm_code': 'OPENQASM 3.0;\nqubit[2] q;\nbit[2] c;\nh q[0];\ncx q[0], q[1];\nc[0] = measure q[0];\nc[1] = measure q[1];\n',
            'backend': 'statevector',
            'shots': 512,
        },
    },
]

cw_results = []
for t in cold_warm_targets:
    print(f'Cold/Warm: {t["label"]} ...')
    cw = cold_vs_warm(t['method'], t['url'], json=t['json'], headers=hdrs, n_warm=10)
    cw['label'] = t['label']
    cw_results.append(cw)
    print(f"  Cold: {cw['cold_ms']:.1f}ms  Warm mean: {cw['warm_mean_ms']:.1f}ms  Speedup: {cw['speedup_factor']:.2f}x")

df_cw = pd.DataFrame(cw_results)
df_cw.to_csv(RESULTS_DIR / 'cold_warm.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(df_cw))
w = 0.35
ax.bar(x - w/2, df_cw['cold_ms'], w, label='Cold (1st hit)', color='salmon')
ax.bar(x + w/2, df_cw['warm_mean_ms'], w, label='Warm (mean)', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(df_cw['label'])
ax.set_ylabel('Latency (ms)')
ax.set_title('Cold vs. Warm Response Time')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'cold_warm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved cold_warm.csv + cold_warm.png')

## 6 — Summary Table

In [ ]:
summary_display = df[['endpoint', 'p50_ms', 'p90_ms', 'p95_ms', 'p99_ms', 'mean_ms', 'error_rate']].copy()
summary_display.columns = ['Endpoint', 'p50 (ms)', 'p90 (ms)', 'p95 (ms)', 'p99 (ms)', 'Mean (ms)', 'Error Rate']
for col in ['p50 (ms)', 'p90 (ms)', 'p95 (ms)', 'p99 (ms)', 'Mean (ms)']:
    summary_display[col] = summary_display[col].round(1)
summary_display['Error Rate'] = summary_display['Error Rate'].map('{:.1%}'.format)
summary_display['✓ < 2s p95'] = df['p95_ms'].apply(lambda x: '✓' if x < 2000 else '✗')
display(summary_display)
print(f"\n{(df['p95_ms'] < 2000).sum()}/{len(df)} endpoints meet the <2s p95 target")